# 学習前の 4B で Perception Test の問題を選別する（Colab A100）

GRPO はグループ内の正誤にばらつきがないと学習信号が出ない。学習前の 4B に 1 問 8 回答えさせ、正解率が 2〜8 割の問題だけを学習に使う。

前提: `MyDrive/movie_llm/pt_frames_f32_s640.zip`（`colab_pt_frames.ipynb` の出力）がある。
ランタイム: A100 GPU。train の 7,360 問 × 8 回で 1〜2 時間の見込み。途中で切れても再実行で続きから進む。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. リポジトリと依存パッケージ

In [ ]:
%cd /content
!rm -rf movie_llm && git clone -q https://github.com/kanaharu20/movie_llm.git
%cd /content/movie_llm
!pip install -q "transformers>=5.0" "accelerate>=1.0" "av>=14" "pillow>=10" "pandas>=2" "pyarrow" "huggingface_hub" "python-dotenv" "openai"
!pip uninstall -y -q torchao  # Colab 既定の古い torchao (0.10) を peft が拒否するので外す
!pip install -q -e . --no-deps

## 2. 問題ファイルを作る（注釈の取得 → 日本語化 → 変換）

In [ ]:
%cd /content/movie_llm
!python scripts/pt_prepare.py
!python scripts/pt_convert.py --splits train --out data/perception_test/items_train.jsonl

## 3. Drive をマウントしてフレームを展開

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/movie_llm'
!mkdir -p /content/pt_frames
!unzip -q -o {DRIVE}/pt_frames_f32_s640.zip -d /content/pt_frames
!ls /content/pt_frames | wc -l
# 予測スクリプトは動画ファイルの有無で対象を決めるので、同名の空ファイルを置く
import os
os.makedirs('/content/pt_videos', exist_ok=True)
for vid in os.listdir('/content/pt_frames'):
    open(f'/content/pt_videos/{vid}.mp4', 'a').close()
print('動画（ダミー）', len(os.listdir('/content/pt_videos')))

## 4. 1 問 8 回答えさせる（結果は Drive に追記。切れたら再実行）

In [ ]:
%cd /content/movie_llm
!mkdir -p {DRIVE}/pt
!python scripts/pt_difficulty.py --items data/perception_test/items_train.jsonl --out {DRIVE}/pt/difficulty_4b_f32.jsonl \
    --model Qwen/Qwen3-VL-4B-Instruct --frames 32 --n 8 \
    --video-dir /content/pt_videos --frames-dir /content/pt_frames

## 5. 集計

In [ ]:
%cd /content/movie_llm
!python scripts/pt_difficulty.py --items data/perception_test/items_train.jsonl --out {DRIVE}/pt/difficulty_4b_f32.jsonl --summarize
print('終わったら手元の私に「選別できた」と伝える。difficulty_4b_f32.jsonl を手元の data/perception_test/ に置く')